Importing packages:

In [1]:
import sys
# Ensure we aren't accidentally pulling from Roaming
sys.path = [p for p in sys.path if "Roaming" not in p]

try:
    from scipy import sparse
    import sklearn
    print("Success! Scipy Sparse and Sklearn are both loaded.")
except ImportError as e:
    print(f"Still failing: {e}")

Success! Scipy Sparse and Sklearn are both loaded.


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
#from langchain_community.chains import LLMChain
import json
from langchain_community.llms import Ollama
from langchain_community.chat_models import ChatOllama
import os
import urllib.request
from pathlib import Path
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
import pandas as pd
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings, OpenAIEmbeddings
#from langchain.evaluation import load_evaluator
from datasets import Dataset
from ragas.metrics.collections import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from datasets import load_dataset
from langchain_core.prompts import ChatPromptTemplate
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextUtilization, ContextPrecision
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.metrics.base import Metric


load_dotenv()

c:\Users\Public\anaconda3\envs\agenticEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0213 07:06:19.761000 13528 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
C:\Users\Namrata Thakur\AppData\Local\Temp\ipykernel_13528\2035638977.py:32: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextUtilization, ContextPrecision
C:\Users\Namrata Thakur\AppData\Local\Temp\ipykernel_13528\2035638977.py:32: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and 

True

In [3]:
import json
from datasets import Dataset
import sys
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
import torch

import os
from dotenv import load_dotenv
load_dotenv()


C:\Users\Namrata Thakur\AppData\Local\Temp\ipykernel_13528\837116289.py:4: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


True

In [ ]:
# print(f"Python Path: {sys.executable}")
# print(f"Torch Path: {torch.__file__}")
# try:
#     import torch._C._dynamo.guards
#     print("Dynamo found!")
# except Exception as e:
#     print(f"Error caught: {e}")

Checking Pytorch and Cuda Version:

In [4]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu130
CUDA available: True
CUDA version: 13.0
GPU: NVIDIA GeForce GTX 1650 Ti


In [5]:
import torch
print(f"PyTorch CUDA: {torch.version.cuda}")
print(torch.cuda.get_device_capability())
!nvcc --version

PyTorch CUDA: 13.0
(7, 5)


'nvcc' is not recognized as an internal or external command,
operable program or batch file.


In [8]:
!nvidia-smi


Thu Feb 12 07:54:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.83                 Driver Version: 581.83         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650 Ti   WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   43C    P8              5W /   30W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Get the base model to be fine-tuned:

In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 2048, 
    load_in_4bit = True,  
    load_in_8bit = False, 
    full_finetuning = False, 
)

==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce GTX 1650 Ti. Num GPUs = 1. Max memory: 4.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Adding the system prompt to the train dataset:

In [7]:
def add_system_prompt(system_prompt, dataset_path):
    data = []
    with open(dataset_path, "r+", encoding="utf8") as f:
        for line in f:
            train_data = json.loads(line)
            sys_message = {"role": "system", "content": system_prompt}
            user_message = train_data.get("messages", [])
            final_message = [sys_message] + user_message
            data.append({"messages" : final_message})

    return data 

In [8]:
data = add_system_prompt(system_prompt="You are a helpful assistant", dataset_path="../data/train_dataset.jsonl")
print("Total Training Examples : ", len(data))
data[90]

Total Training Examples :  92


{'messages': [{'role': 'system', 'content': 'You are a helpful assistant'},
  {'role': 'user',
   'content': "What happens to the model's performance when trained naïvely without dedicated modules?"},
  {'role': 'assistant',
   'content': 'The model suffers from severe catastrophic forgetting, with performance dropping to zero for earlier splits and FPP rising to 56.82.'}]}

In [9]:
train_dataset = Dataset.from_list(data)
train_dataset

Dataset({
    features: ['messages'],
    num_rows: 92
})

Converting the base model to PEFT enabled model

In [10]:
model = FastLanguageModel.get_peft_model(
    model=model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None
)
print("PEFT enabled LLAMA3.2:1b model ready..!")

Unsloth 2026.2.1 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


PEFT enabled LLAMA3.2:1b model ready..!


Formatting train data as suitable for LLAMA:

In [11]:
def train_dataset_formatting(batch):
    messages = batch['messages']
    formatted_message = [
        tokenizer.apply_chat_template(mess,tokenize=False, add_generation_prompt=False) for mess in messages
    ]
    return {"text": formatted_message}

train_dataset = train_dataset.map(train_dataset_formatting,batched=True)
print(train_dataset)
train_dataset[90]

Map: 100%|██████████| 92/92 [00:00<00:00, 2681.72 examples/s]

Dataset({
    features: ['messages', 'text'],
    num_rows: 92
})


{'messages': [{'content': 'You are a helpful assistant', 'role': 'system'},
  {'content': "What happens to the model's performance when trained naïvely without dedicated modules?",
   'role': 'user'},
  {'content': 'The model suffers from severe catastrophic forgetting, with performance dropping to zero for earlier splits and FPP rising to 56.82.',
   'role': 'assistant'}],
 'text': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 13 Feb 2026\n\nYou are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat happens to the model's performance when trained naïvely without dedicated modules?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nThe model suffers from severe catastrophic forgetting, with performance dropping to zero for earlier splits and FPP rising to 56.82.<|eot_id|>"}

Fine-Tuning the model:

In [12]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=None,

    args= SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=80, #100 --> overfitting
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=2025,
        report_to="none"

    ),
)


Unsloth: Tokenizing ["text"]: 100%|██████████| 92/92 [00:00<00:00, 2117.51 examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [13]:
#Taken from https://medium.com/mitb-for-all/how-to-train-your-llm-teaching-toothless-to-bite-8d9f56fe4b2a
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce GTX 1650 Ti. Max memory = 4.0 GB.
1.564 GB of memory reserved.


In [14]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 92 | Num Epochs = 7 | Total steps = 80
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
1,4.638200
2,5.351100
3,4.855900
4,4.168500
5,4.022100
6,3.682800
7,3.017600
8,2.737400
9,2.414700
10,2.242100


Stopped the training as loss is good now..!

In [15]:
#Taken from https://medium.com/mitb-for-all/how-to-train-your-llm-teaching-toothless-to-bite-8d9f56fe4b2a
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

1465.869 seconds used for training.
24.43 minutes used for training.
Peak reserved memory = 1.658 GB.
Peak reserved memory for training = 0.094 GB.
Peak reserved memory % of max memory = 41.45 %.
Peak reserved memory for training % of max memory = 2.35 %.


In [ ]:
#This is merge LoRA adapters with model weights before pushing to Huggingface Hub:
model.push_to_hub_merged("NamrataThakur/llama32-1bn_finetuned", tokenizer, 
                         save_method = "merged_16bit", token = os.environ["HF_TOKEN"])
#tokenizer.push_to_hub("NamrataThakur/llama32-1bn_finetuned", token = os.environ["HF_TOKEN"])

tokenizer.json: 100%|██████████| 17.2M/17.2M [00:04<00:00, 3.93MB/s]


Found HuggingFace hub cache directory: C:\Users\Namrata Thakur\.cache\huggingface\hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [02:54<00:00, 174.10s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


model.safetensors: 100%|██████████| 2.47G/2.47G [03:17<00:00, 12.5MB/s]/s]
Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [03:34<00:00, 214.67s/it]


Unsloth: Merge process complete. Saved to `d:\LLM_Deeplearning.ai\Fine-tuning-LLMs-Strategies\notebooks\NamrataThakur\llama32-1bn_finetuned`


In [16]:
#This is merge LoRA adapters with model weights before saving in the local directory:
model.save_pretrained_merged("llama32-1bn_finetuned_c", tokenizer, save_method = "merged_16bit")

Found HuggingFace hub cache directory: C:\Users\Namrata Thakur\.cache\huggingface\hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [02:53<00:00, 173.13s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:09<00:00,  9.41s/it]


Unsloth: Merge process complete. Saved to `d:\LLM_Deeplearning.ai\Fine-tuning-LLMs-Strategies\notebooks\llama32-1bn_finetuned_c`


In [ ]:
#ERROR IN GETTING GGUF FORM --> llama.cpp error

# model.save_pretrained_gguf(
#     "llama32-1bn_finetuned", 
#     tokenizer, 
#     quantization_method = "f16" 
# )

In [ ]:
#SAME ERROR AS ABOVE:
# model.push_to_hub_gguf("NamrataThakur/llama32-1bn_finetuned", tokenizer=tokenizer, 
#                        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
#                        token = os.environ["HF_TOKEN"])

Since llama.cpp was giving some issueswith binaries, I shift to manual conversion to GGUF using the "convert_hf_to_gguf.py" file present in llama.cpp folder:

Save the model as a large F16 file and then use Ollama's internal engine (ollama create) to shrink it (if needed). This avoided the above issues.

In [17]:
!python llama.cpp/convert_hf_to_gguf.py ./llama32-1bn_finetuned_c --outfile llama32-1bn_c_f16.gguf --outtype f16

INFO:hf-to-gguf:Loading model: llama32-1bn_finetuned_c
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {32}
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> F16, shape = {2048, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> F16, shape = {8192, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> F16, shape = {2048, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> F16, shape = {2048, 8192}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.bfloat16 --> F16, shape = {2048, 512}
INFO:hf-to-gguf:blk.0.attn

In [ ]:
# Command: ollama create <model_name> -f Modelfile -q <quantised_version>
!ollama create llama32-1bn-customFT -f Modelfile -q f16

In [18]:
!ollama list

NAME                                ID              SIZE      MODIFIED       
llama32-1bn-customFT:latest         4acd28be8e22    2.5 GB    17 seconds ago    
llama32-1bn_f16_finetuned:latest    e79c6e7e9458    1.3 GB    10 hours ago      
nomic-embed-text:latest             0a109f422b47    274 MB    46 hours ago      
llama3.2:1b                         baf6a787fdff    1.3 GB    47 hours ago      
llama3:latest                       365c0bd3c000    4.7 GB    9 months ago      


EVALUATION

In [19]:
#Check the fine-tuned model name in "ollama list".
ft_model = ChatOllama(
    model="llama32-1bn-customFT", #Give the same model name that is used in "ollama create" command
    temperature=0.01
)

# Prompt used to get answers on the model that we are aiming to fine-tune:
ft_prompt = ChatPromptTemplate.from_template(
    template="""
You are answering a question using retrieved context from a document.

Instructions:
- Use ONLY the information in the context.
- You MAY paraphrase, summarize, and combine information.
- If the context provides partial information, answer as fully as possible.
- Do NOT introduce facts not supported by the context.
- Only say "I don't know" if the context gives no useful information at all.

Context:
{context}

Question:
{question}
"""
)

ft_chain = ft_prompt | ft_model

C:\Users\Namrata Thakur\AppData\Local\Temp\ipykernel_13528\91044508.py:2: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  ft_model = ChatOllama(


In [21]:
def generate_answers(dataset, chain):
    test_df = []
    for example in dataset:
        res = chain.invoke(input={
            'question': example['question'],
            'context': example['context']
        })
        response = res.content
        obj = {
            'question' : example['question'],
            'contexts' : example['context'],
            'ground_truth' : example['answer'],
            'sft_fineTuned_answers' : response
        }
        test_df.append(obj)

    test_generated_dataset = pd.DataFrame.from_records(data=test_df)
    return test_generated_dataset

In [22]:
test_dataset = pd.read_csv("../data/test_dataset.csv")
test_examples = test_dataset.to_dict(orient='records')

In [23]:
test_examples[0]

{'question': 'What is the average precision (mAP) achieved by the CLIMB-3D method in the Split-A scenario?',
 'answer': '22.72',
 'context': 'Table 1: Comparison between the proposed method and the baseline in the 3DIS setting, evaluated using mAP25, mAP50, mAP, and FPP after training across all phase. Scenarios Methods Average Precision↑FPP↓ mAP25 mAP50 mAP mAP 25 mAP50 Baseline 16.46 14.29 10.44 51.30 46.82Split-A CLIMB-3D(Ours) 35.69 31.05 22.72 3.44 2.63 Baseline 17.22 15.07 10.93 46.27 42.1Split-B CLIMB-3D(Ours) 35.48 31.56 23.69 8.00 5.51 Baseline 25.65 21.08 14.85 31.68 28.84Split-C CLIMB-3D(Ours) 31.59 26.78 18.93 9.10 7.89 To evaluate our proposed approach, we construct a baseline using ER [5] for the 3DIS setting, as no existing incremental baselines are available for this task.'}

In [24]:
test_generated_dataset = generate_answers(test_examples, ft_chain)
test_generated_dataset.head()

,question,contexts,ground_truth,sft_fineTuned_answers
0,What is the average precision (mAP) achieved b...,Table 1: Comparison between the proposed metho...,22.72,I don't know
1,What new object categories are introduced in T...,2THENGANE ET AL.: CLIMB-3D 1 Introduction Tabl...,"Pillow, Coffee Table, and Sofa Chair.",I don't know
2,What is the effect of adding exemplar replay (...,10THENGANE ET AL.: CLIMB-3D Effect of ER.Addin...,Exemplar replay (ER) yields notable gains for ...,"A) Increases by +18.51% and +10.38%, respectiv..."
3,What evaluation metrics are used to assess the...,8THENGANE ET AL.: CLIMB-3D assessing real-worl...,Mean Average Precision (mAP) and mean Intersec...,Answer:\nMean Average Precision (mAP) and Mean...
4,What is one of the main contributions of Theng...,"THENGANE ET AL.: CLIMB-3D3 In summary, our con...",A novel problem setting of imbalanced class-in...,A) A method for balancing learning and forgett...


In [25]:
test_generated_dataset.rename(columns={'sft_fineTuned_answers':'answer'},inplace=True)
test_generated_dataset.to_csv("../data/testEvalData_sft-FineTuned.csv", index=False)

In [ ]:
test_generated_dataset = pd.read_csv("../data/testEvalData_sft-FineTuned.csv")

In [27]:
def get_rag_evaluation(embed_name, dataset, model_name):
    
    
    ragas_data = []
    for _, row in dataset.iterrows():
        
        obj ={
            'question' : row['question'],
            'contexts' : [row['contexts']],
            'ground_truth' : row['ground_truth'],
            'answer' : row['answer']
            
        }
        ragas_data.append(obj)
    
    data = Dataset.from_list(ragas_data)

    # 1. Setup an OpenAI-compatible client for Ollama
    # Ollama provides an OpenAI-compatible endpoint at /v1 --> If any other model to be used here:
    client = OpenAI()

    # 2. Use the llm_factory instead of LangchainLLMWrapper
    # Use provider="openai" to use the compatible client
    #LLM given here is the judge. So, use the same LLM that is used to generate this synthetic data
    evaluator_llm = llm_factory(model_name, client=client)

    #Throwing some errors so switched to LangchainEmbeddingsWrapper:
    # evaluator_embeddings = embedding_factory("openai", model=embed_name, client=client)

    # lc_embeddings = OllamaEmbeddings(
    # model=embed_name  # e.g. "nomic-embed-text"
    # )
    lc_embeddings = OpenAIEmbeddings(model=embed_name)
    ragas_embeddings = LangchainEmbeddingsWrapper(lc_embeddings)
    
    #Defining the metrics
    m1 = Faithfulness(llm=evaluator_llm)
    m2 = AnswerRelevancy(llm=evaluator_llm, embeddings=ragas_embeddings)
    m3 = AnswerCorrectness(llm=evaluator_llm, embeddings=ragas_embeddings)
    m4 = ContextPrecision(llm=evaluator_llm)

    metric_list = [m1, m2, m3, m4]
    
    results = evaluate(dataset=data,
                       metrics=metric_list,
                       embeddings=ragas_embeddings,
                        )
    eval_df = results.to_pandas()
    return eval_df

In [28]:
# nomic-embed-text
eval_df = get_rag_evaluation(embed_name="text-embedding-ada-002", dataset=test_generated_dataset, 
                             model_name="gpt-4o-mini") #Using the same model that we used earlier to generate the synthetic data

C:\Users\Namrata Thakur\AppData\Local\Temp\ipykernel_13528\4004107200.py:33: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import OpenAIEmbeddings``.
  lc_embeddings = OpenAIEmbeddings(model=embed_name)
C:\Users\Namrata Thakur\AppData\Local\Temp\ipykernel_13528\4004107200.py:34: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(lc_embeddings)
Evaluating:   0%|          | 0/96 [00:00<?, ?it/s][ragas.prompt.pydantic_promp

In [29]:
eval_df.head()

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,answer_correctness,context_precision
0,What is the average precision (mAP) achieved b...,[Table 1: Comparison between the proposed meth...,I don't know,22.72,0.00,0.000000,0.191037,1.0
1,What new object categories are introduced in T...,[2THENGANE ET AL.: CLIMB-3D 1 Introduction Tab...,I don't know,"Pillow, Coffee Table, and Sofa Chair.",0.00,0.000000,0.189038,1.0
2,What is the effect of adding exemplar replay (...,[10THENGANE ET AL.: CLIMB-3D Effect of ER.Addi...,"A) Increases by +18.51% and +10.38%, respectiv...",Exemplar replay (ER) yields notable gains for ...,0.25,0.780958,0.337955,1.0
3,What evaluation metrics are used to assess the...,[8THENGANE ET AL.: CLIMB-3D assessing real-wor...,Answer:\nMean Average Precision (mAP) and Mean...,Mean Average Precision (mAP) and mean Intersec...,1.00,0.872557,0.993288,1.0
4,What is one of the main contributions of Theng...,"[THENGANE ET AL.: CLIMB-3D3 In summary, our co...",A) A method for balancing learning and forgett...,A novel problem setting of imbalanced class-in...,1.00,0.754763,0.444135,1.0


Supervised Fine-Tuned Model Scores on Evaluation Data:

In [30]:
mean_faithfulness_score = eval_df['faithfulness'].mean()
mean_answer_relevancy_score = eval_df['answer_relevancy'].mean()
mean_answer_correctness_score = eval_df['answer_correctness'].mean()
print("Mean Faithfulness Score : ", mean_faithfulness_score)
print("Mean Answer Relevancy Score :", mean_answer_relevancy_score)
print("Mean Answer Correctness Score :", mean_answer_correctness_score)

Mean Faithfulness Score :  0.3541666666666667
Mean Answer Relevancy Score : 0.5572119598860267
Mean Answer Correctness Score : 0.49001781944572986


Saving the scores

In [31]:
eval_df.to_csv('../data/sft-FineTuned_evaluationScores.csv', index=False)